# Quantize `final_model.keras` sang INT8 TFLite và mảng C++

Chạy lần lượt từ đầu, với working directory là **gốc repository**. Notebook dùng `saved_models_cnn_lstm/final_model.keras`, `scaler.pkl`, `metadata.pkl` và dữ liệu `kfall/` gốc. Không huấn luyện lại model.

Pipeline: kiểm tra artifacts → tạo window K-Fall giống `final_model.ipynb` → hiệu chuẩn INT8 → kiểm tra tensor/độ lệch dự đoán → xuất `.tflite`, `model_data.cc`, `model_data.h` và manifest. Output nằm trong `saved_models_cnn_lstm/quantized/`; **không ghi đè firmware ESP32**.

Các window K-Fall đã góp phần train `final_model.keras`, nên phép so sánh dưới đây chỉ kiểm tra **độ trung thực của chuyển đổi**, không phải đánh giá khả năng tổng quát hóa. Muốn đánh giá trên người mới, cần bộ dữ liệu độc lập; muốn báo cáo metric INT8 theo LOSO, phải quantize từng checkpoint của từng fold bằng dữ liệu train tương ứng.


In [ ]:
from pathlib import Path
import hashlib
import json
import re
import zipfile
import xml.etree.ElementTree as ET

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from ai_edge_litert.interpreter import Interpreter

ROOT = Path.cwd()
MODEL_DIR = ROOT / "saved_models_cnn_lstm"
MODEL_PATH = MODEL_DIR / "final_model.keras"
SCALER_PATH = MODEL_DIR / "scaler.pkl"
METADATA_PATH = MODEL_DIR / "metadata.pkl"
LABEL_DIR = ROOT / "kfall" / "label_data_new"
SENSOR_DIR = ROOT / "kfall" / "sensor_data_new"
OUTPUT_DIR = MODEL_DIR / "quantized"

# 256 window/lớp cho calibration; 128 window/lớp để đối chiếu Keras và TFLite.
CALIBRATION_PER_CLASS = 256
COMPARISON_PER_CLASS = 128
RANDOM_SEED = 42
MAX_MEAN_ABS_ERROR = 0.05
MIN_LABEL_AGREEMENT = 0.95

for path in (MODEL_PATH, SCALER_PATH, METADATA_PATH, LABEL_DIR, SENSOR_DIR):
    if not path.exists():
        raise FileNotFoundError(f"Thiếu {path}; chạy notebook từ gốc repository và chuẩn bị K-Fall.")

print("TensorFlow:", tf.__version__)
print("Model:", MODEL_PATH)


## 1. Nạp và kiểm tra đúng bộ artifacts

Chỉ nạp các file `joblib/pickle` do chính project tạo ra vì định dạng này có thể thực thi mã khi mở file không tin cậy.


In [ ]:
model = tf.keras.models.load_model(MODEL_PATH, compile=False)
scaler = joblib.load(SCALER_PATH)
metadata = joblib.load(METADATA_PATH)

features = list(metadata["feature_cols"])
window_size = int(metadata["window_size"])
window_step = int(metadata["window_step"])
fall_ratio_threshold = float(metadata["fall_ratio_threshold"])
decision_threshold = float(metadata["decision_threshold"])

if features != ["AccX", "AccY", "AccZ"]:
    raise ValueError(f"Notebook đang dành cho final model Acc XYZ; metadata ghi {features}")
if (window_size, window_step, fall_ratio_threshold) != (200, 100, 0.20):
    raise ValueError("Window/step/nhãn không khớp cấu hình final_model.ipynb đã đọc.")
if tuple(model.input_shape) != (None, window_size, len(features)) or tuple(model.output_shape) != (None, 1):
    raise ValueError(f"Shape model không khớp metadata: {model.input_shape}, {model.output_shape}")
if len(scaler.mean_) != len(features) or len(scaler.scale_) != len(features):
    raise ValueError("Scaler không khớp số feature.")
if not np.isfinite(scaler.mean_).all() or not np.isfinite(scaler.scale_).all() or np.any(scaler.scale_ <= 0):
    raise ValueError("Scaler không hợp lệ.")

print("Input:", model.input_shape, "Output:", model.output_shape)
print("Features:", features, "window step:", window_step, "threshold:", decision_threshold)
print("Scaler mean:", scaler.mean_)
print("Scaler scale:", scaler.scale_)


## 2. Lấy window đại diện từ K-Fall

Tái lập quy tắc gán nhãn trong notebook gốc: `FrameCounter` từ onset đến impact (kể cả hai đầu); window 200 mẫu, step 100, lớp Fall nếu ít nhất 20% frame là fall. Lấy mẫu reservoir theo lớp trên toàn bộ file để calibration có cả chuyển động thường và ngã. Tập đối chiếu không trùng window calibration, nhưng vẫn thuộc dữ liệu đã train model cuối.


In [ ]:
subjects = list(metadata["subjects"])
if len(subjects) != 32:
    raise ValueError(f"Metadata cần 32 subject, nhận {len(subjects)}")

def read_label_rows(label_path):
    try:
        df = pd.read_excel(label_path)
        task_col = "Task Code (Task ID)"
        required = [task_col, "Trial ID", "Fall_onset_frame", "Fall_impact_frame"]
        if not set(required).issubset(df.columns):
            raise ValueError(f"Thiếu cột nhãn trong {label_path}")
        df[task_col] = df[task_col].ffill()
        for _, row in df.dropna(subset=required).iterrows():
            yield str(row[task_col]), int(row["Trial ID"]), int(row["Fall_onset_frame"]), int(row["Fall_impact_frame"])
    except ImportError:
        # Fallback giống final_model.ipynb khi venv không cài openpyxl.
        with zipfile.ZipFile(label_path) as archive:
            tree = ET.fromstring(archive.read("xl/worksheets/sheet1.xml"))
        current_task = ""
        for row in tree.findall(".//{*}row")[1:]:
            values = {}
            for cell in row.findall(".//{*}c"):
                col = "".join(ch for ch in cell.get("r", "") if ch.isalpha())
                inline = cell.find(".//{*}t")
                value = cell.find(".//{*}v")
                values[col] = (inline.text if inline is not None else value.text if value is not None else "") or ""
            if values.get("A"):
                current_task = values["A"].strip()
            if all(values.get(col, "").strip() for col in ("C", "D", "E")):
                yield current_task, int(float(values["C"])), int(float(values["D"])), int(float(values["E"]))

label_lookup = {}
for subject in subjects:
    label_path = LABEL_DIR / f"{subject}_label.xlsx"
    if not label_path.exists():
        raise FileNotFoundError(f"Thiếu nhãn {label_path}; tránh gán nhầm fall thành normal.")
    for task, trial, onset, impact in read_label_rows(label_path):
        match = re.search(r"\((\d+)\)", task)
        if match:
            key = (subject, int(match.group(1)), trial)
            label_lookup[key] = (onset, impact)

needed = CALIBRATION_PER_CLASS + COMPARISON_PER_CLASS
rng = np.random.default_rng(RANDOM_SEED)
samples = {0: [], 1: []}
seen = {0: 0, 1: 0}
filename_pattern = re.compile(r"^S(\d{2})T(\d{2})R(\d{2})\.csv$")

for subject in subjects:
    subdir = SENSOR_DIR / subject
    if not subdir.is_dir():
        raise FileNotFoundError(subdir)
    for csv_path in sorted(subdir.glob("*.csv")):
        match = filename_pattern.fullmatch(csv_path.name)
        if not match:
            continue
        task_id, trial_id = int(match.group(2)), int(match.group(3))
        df = pd.read_csv(csv_path, usecols=["FrameCounter", *features])
        raw = df[features].to_numpy(dtype=np.float32)
        frames = df["FrameCounter"].to_numpy()
        if not np.isfinite(raw).all():
            raise ValueError(f"Giá trị sensor không hữu hạn: {csv_path}")
        interval = label_lookup.get((subject, task_id, trial_id))
        fall_mask = ((frames >= interval[0]) & (frames <= interval[1])) if interval else np.zeros(len(frames), dtype=bool)
        for start in range(0, len(raw) - window_size + 1, window_step):
            end = start + window_size
            label = int(np.mean(fall_mask[start:end]) >= fall_ratio_threshold)
            seen[label] += 1
            bucket = samples[label]
            if len(bucket) < needed:
                bucket.append(raw[start:end].copy())
            else:
                replacement = int(rng.integers(0, seen[label]))
                if replacement < needed:
                    bucket[replacement] = raw[start:end].copy()

if any(len(samples[label]) < needed for label in (0, 1)):
    raise ValueError(f"Không đủ window mỗi lớp: cần {needed}, đã thấy {seen}")

def scale_windows(windows):
    windows = np.asarray(windows, dtype=np.float32)
    return scaler.transform(windows.reshape(-1, len(features))).reshape(-1, window_size, len(features)).astype(np.float32)

calibration = scale_windows(samples[0][:CALIBRATION_PER_CLASS] + samples[1][:CALIBRATION_PER_CLASS])
comparison = scale_windows(samples[0][CALIBRATION_PER_CLASS:] + samples[1][CALIBRATION_PER_CLASS:])
comparison_labels = np.array([0] * COMPARISON_PER_CLASS + [1] * COMPARISON_PER_CLASS, dtype=np.int8)
if not np.isfinite(calibration).all() or not np.isfinite(comparison).all():
    raise ValueError("Window sau StandardScaler có giá trị không hữu hạn")
print("Số window đã thấy:", seen)
print("Calibration:", calibration.shape, "Đối chiếu:", comparison.shape)


## 3. Chuyển đổi với full integer INT8

`representative_dataset` nhận window **đã qua scaler**, dạng float32 `(1, 200, 3)`. Với TensorFlow 2.21 trong venv hiện tại, chuyển trực tiếp LSTM bị lỗi `TensorListReserve`; vì vậy clone model với `LSTM(unroll=True)`, copy nguyên trọng số và kiểm tra đầu ra float trước khi convert. Đây chỉ là đổi cách biểu diễn phép tính, không huấn luyện lại. Chỉ chấp nhận built-in INT8 và input/output INT8; không bật `SELECT_TF_OPS`.


In [ ]:
def representative_dataset():
    for window in calibration:
        yield [window[np.newaxis, ...]]

def clone_with_unrolled_lstm(layer):
    config = layer.get_config()
    if isinstance(layer, tf.keras.layers.LSTM):
        config["unroll"] = True
    return layer.__class__.from_config(config)

conversion_model = tf.keras.models.clone_model(model, clone_function=clone_with_unrolled_lstm)
conversion_model.set_weights(model.get_weights())
reference = np.asarray(model(calibration[:16], training=False))
unrolled = np.asarray(conversion_model(calibration[:16], training=False))
unroll_error = float(np.max(np.abs(reference - unrolled)))
print("Max difference khi unroll LSTM:", unroll_error)
if unroll_error > 1e-5:
    raise ValueError("Unrolled LSTM không khớp model gốc; dừng chuyển đổi.")

converter = tf.lite.TFLiteConverter.from_keras_model(conversion_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

try:
    tflite_bytes = converter.convert()
except Exception as exc:
    raise RuntimeError(
        "Không chuyển được full INT8. Kiểm tra LSTM/ops và phiên bản TensorFlow; "
        "không xuất .cc từ model chưa tương thích."
    ) from exc

print("Kích thước TFLite:", len(tflite_bytes), "bytes")


## 4. Kiểm tra kiểu tensor và phép toán

Tensor hình dạng (`int32`/`int64`) có thể xuất hiện và không phải phép toán float. Điều kiện ở đây là không có tensor `float32/float64`, đồng thời input/output đều INT8. Danh sách op được in ra để đối chiếu với resolver và kernel của bản TFLite Micro trên ESP32-S3.


In [ ]:
interpreter = Interpreter(model_content=tflite_bytes, experimental_preserve_all_tensors=True)
interpreter.allocate_tensors()
inputs = interpreter.get_input_details()
outputs = interpreter.get_output_details()
if len(inputs) != 1 or len(outputs) != 1:
    raise ValueError("Cần đúng một input và một output")
input_info, output_info = inputs[0], outputs[0]
if input_info["dtype"] != np.int8 or output_info["dtype"] != np.int8:
    raise ValueError("Input/output chưa phải INT8")
if tuple(input_info["shape"]) != (1, window_size, len(features)) or tuple(output_info["shape"]) != (1, 1):
    raise ValueError(f"Shape TFLite bất ngờ: {input_info['shape']}, {output_info['shape']}")
floating_tensors = [t["name"] for t in interpreter.get_tensor_details() if np.issubdtype(t["dtype"], np.floating)]
if floating_tensors:
    raise ValueError(f"Model còn tensor float: {floating_tensors[:10]}")
input_scale, input_zero_point = input_info["quantization"]
output_scale, output_zero_point = output_info["quantization"]
if input_scale <= 0 or output_scale <= 0:
    raise ValueError("Scale lượng tử hóa không hợp lệ")

# API private này chỉ để xem tên op; khả năng chạy trên ESP32 vẫn phải kiểm tra bằng TFLite Micro.
op_names = sorted({op["op_name"] for op in interpreter._get_ops_details()} - {"DELEGATE"})
micro_blockers = sorted(set(op_names) & {"VAR_HANDLE", "READ_VARIABLE", "ASSIGN_VARIABLE", "CALL_ONCE", "FlexTensorListReserve", "FlexTensorListStack"})
print("Input scale/zero-point:", input_scale, input_zero_point)
print("Output scale/zero-point:", output_scale, output_zero_point)
print("Ops:", op_names)
if micro_blockers:
    print("CẢNH BÁO: op cần xử lý/kiểm chứng trên TFLite Micro trước khi tích hợp ESP32:", micro_blockers)


## 5. So sánh dự đoán với model Keras

Input INT8: `clip(round(x_scaled / input_scale) + input_zero_point, -128, 127)`. Output xác suất: `(q_output - output_zero_point) × output_scale`. So sánh trên các window không dùng cho hiệu chuẩn, nhưng vẫn thuộc tập train model cuối. Hai ngưỡng bên dưới là **gate kỹ thuật có thể chỉnh**, không phải chứng nhận chất lượng lâm sàng hay metric LOSO.


In [ ]:
keras_scores = np.asarray(model.predict(comparison, batch_size=32, verbose=0)).reshape(-1)
int8_scores = []
for window in comparison:
    q_input = np.clip(np.rint(window / input_scale) + input_zero_point, -128, 127).astype(np.int8)
    interpreter.set_tensor(input_info["index"], q_input[np.newaxis, ...])
    interpreter.invoke()
    q_output = int(interpreter.get_tensor(output_info["index"]).reshape(-1)[0])
    int8_scores.append((q_output - output_zero_point) * output_scale)
int8_scores = np.asarray(int8_scores, dtype=np.float32)

if not np.isfinite(keras_scores).all() or not np.isfinite(int8_scores).all():
    raise ValueError("Dự đoán có giá trị không hữu hạn")
abs_error = np.abs(keras_scores - int8_scores)
keras_pred = keras_scores >= decision_threshold
int8_pred = int8_scores >= decision_threshold
agreement = float(np.mean(keras_pred == int8_pred))
mean_abs_error = float(np.mean(abs_error))

print(f"Mean abs error: {mean_abs_error:.6f}; max abs error: {float(np.max(abs_error)):.6f}")
print(f"Agreement tại ngưỡng {decision_threshold}: {agreement:.2%}; đổi nhãn: {int(np.sum(keras_pred != int8_pred))}/{len(comparison)}")
print("Nhãn K-Fall trong tập đối chiếu:", np.bincount(comparison_labels, minlength=2).tolist())
if mean_abs_error > MAX_MEAN_ABS_ERROR or agreement < MIN_LABEL_AGREEMENT:
    raise RuntimeError(
        f"Sai khác vượt gate (MAE <= {MAX_MEAN_ABS_ERROR}, agreement >= {MIN_LABEL_AGREEMENT}). "
        "Xem lại hiệu chuẩn, preprocessing và kiến trúc trước khi xuất artifact."
    )


## 6. Xuất `.tflite`, `.cc`, `.h` và manifest

`.cc` chỉ là mảng byte của file TFLite, chưa đồng nghĩa firmware chạy được. Không tự động ghi đè `esp32s3-tinyml-fall-detector/main/model/model_data.cc`: firmware hiện tại dùng `(200, 6)` còn model này dùng `(200, 3)`. File chuyển từ LSTM unrolled có thể chứa `VAR_HANDLE`/`READ_VARIABLE`/`CALL_ONCE`, những op chưa có trong resolver firmware hiện tại. Khi tích hợp cần sửa scaler, feature order, input shape, danh sách op, arena và kiểm tra trên ESP32-S3; có thể phải thay kiến trúc nếu runtime không hỗ trợ.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
tflite_path = OUTPUT_DIR / "final_model_int8.tflite"
cc_path = OUTPUT_DIR / "model_data.cc"
h_path = OUTPUT_DIR / "model_data.h"
manifest_path = OUTPUT_DIR / "quantization_manifest.json"

sha256 = hashlib.sha256(tflite_bytes).hexdigest()
tflite_path.write_bytes(tflite_bytes)
h_path.write_text(
    "#ifndef MODEL_DATA_H_\n#define MODEL_DATA_H_\n\n"
    "extern const unsigned char g_fall_model[];\n"
    "extern const unsigned int g_fall_model_len;\n\n"
    "#endif  // MODEL_DATA_H_\n",
    encoding="utf-8",
)
rows = [", ".join(f"0x{value:02x}" for value in tflite_bytes[i:i + 12]) for i in range(0, len(tflite_bytes), 12)]
cc_path.write_text(
    '#include "model_data.h"\n\n'
    f"// Full INT8 TFLite; SHA256: {sha256}\n"
    "alignas(16) const unsigned char g_fall_model[] = {\n"
    + "".join("    " + row + ",\n" for row in rows)
    + "};\n"
    + "const unsigned int g_fall_model_len = sizeof(g_fall_model);\n",
    encoding="utf-8",
)
manifest = {
    "source_model": str(MODEL_PATH.relative_to(ROOT)),
    "tflite_sha256": sha256,
    "tflite_bytes": len(tflite_bytes),
    "feature_cols": features,
    "window_size": window_size,
    "window_step": window_step,
    "decision_threshold": decision_threshold,
    "scaler_mean": [float(v) for v in scaler.mean_],
    "scaler_scale": [float(v) for v in scaler.scale_],
    "input_shape": [int(v) for v in input_info["shape"]],
    "input_scale": float(input_scale),
    "input_zero_point": int(input_zero_point),
    "output_shape": [int(v) for v in output_info["shape"]],
    "output_scale": float(output_scale),
    "output_zero_point": int(output_zero_point),
    "operators": op_names,
    "potential_micro_blockers": micro_blockers,
    "unrolled_lstm_float_max_difference": unroll_error,
    "comparison_mae": mean_abs_error,
    "comparison_max_abs_error": float(np.max(abs_error)),
    "comparison_label_agreement": agreement,
    "comparison_windows": len(comparison),
    "note": "Comparison uses K-Fall windows from final model training; not independent validation or LOSO.",
}
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
if hashlib.sha256(tflite_path.read_bytes()).hexdigest() != sha256:
    raise IOError("SHA256 artifact không khớp")
print("Đã xuất:", tflite_path, cc_path, h_path, manifest_path, sep="\n  ")
print("Bước tiếp: kiểm tra ops/arena/runtime thực tế trên ESP32-S3 trước khi thay firmware.")
